In [0]:
from pyspark.sql import functions as F



In [0]:
%sql
describe external location `data-zone`

In [0]:
base_dir_data = spark.sql("Describe external location `data-zone`").select("url").collect()[0][0]
display(base_dir_data)

In [0]:
base_dir_checkpoint = spark.sql("Describe external location `checkpoint_zone`").select("url").collect()[0][0]
display(base_dir_checkpoint)

In [0]:
Landing_zone = base_dir_data + "/raw"
checkpoint_zone = base_dir_checkpoint + "/checkpoints"
print(f"Landing zone: {Landing_zone}")
print(f"Checkpoint zone: {checkpoint_zone}")

In [0]:
db_name = "sbit_db"
catalog = "dev"

print(f"Creating the database {catalog}.{db_name}...", end='')
spark.sql(f"use {catalog}.{db_name}")


In [0]:
print(f"Creating users table...", end='')
spark.sql(f"""CREATE OR REPLACE TABLE {catalog}.{db_name}.users(
    user_id bigint,
    device_id bigint,
    mac_address string,
    registration_timestamp timestamp
    )  
    """) 


print("Done")

In [0]:
print(f"Creating gym_logs table...", end='')
spark.sql(f"""CREATE OR REPLACE TABLE {catalog}.{db_name}.gym_logs(
    mac_address string,
    gym bigint,
    login timestamp,
    logout timestamp
    )  
    """)
print("Done")

In [0]:
print(f"Creating user_profile table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.user_profile(
    user_id bigint,
    dob  DATE,
    sex STRING,
    first_name STRING,
    last_name STRING,
    street_address STRING,
    city STRING,
    state STRING,
    zip INT,
    updated TIMESTAMP)
    """)
print("Done")

In [0]:
print(f"Creating heart_rate table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.heart_rate(
        device_id LONG, 
        time TIMESTAMP, 
        heartrate DOUBLE, 
        valid BOOLEAN)
        """)
print("Done")

In [0]:
print(f"Creating user_bins table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.user_bins(
        user_id BIGINT, 
        age STRING, 
        gender STRING, 
        city STRING, 
        state STRING)
        """)  
print("Done")

In [0]:
print(f"Creating workouts table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.workouts(
        user_id INT, 
        workout_id INT, 
        time TIMESTAMP, 
        action STRING, 
        session_id INT)
        """)  
print("Done")

In [0]:
print(f"Creating completed_workouts table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.completed_workouts(
        user_id INT, 
        workout_id INT, 
        session_id INT, 
        start_time TIMESTAMP, 
        end_time TIMESTAMP)
        """)  
print("Done")

In [0]:
print(f"Creating workout_bpm table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.workout_bpm(
        user_id INT, 
        workout_id INT, 
        session_id INT,
        start_time TIMESTAMP, 
        end_time TIMESTAMP,
        time TIMESTAMP, 
        heartrate DOUBLE)
        """)  
print("Done")

 **History Loader**

In [0]:
landing_zone = base_dir_data + "/raw"      
test_data_dir = base_dir_data + "/test_data"

In [0]:
print(f"Creating date_lookup table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.date_lookup(
        date date, 
        week int, 
        year int, 
        month int, 
        dayofweek int, 
        dayofmonth int, 
        dayofyear int, 
        week_part string)
        """)  
print("Done")

In [0]:
print('Loading data_lookup table...', end='')
spark.sql(f"""INSERT OVERWRITE TABLE {catalog}.{db_name}.date_lookup
SELECT date, week, year, month, dayofweek, dayofmonth, dayofmonth, dayofyear, week_part FROM json.`{test_data_dir}/6-date-lookup.json/`""")
print('Done')


**INGEST RAW DATA INTO THE BRONZE DELTA TABLE**

In [0]:
once = True

schema = "user_id long, device_id long, mac_address string, registration_timestamp double"

df_stream = (spark.readStream
            .format("cloudFiles")
            .schema(schema)
            .option("maxFilesPerTrigger", 1)
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .load(landing_zone + "/registered_users_bz")
            .withColumn("load_time", F.current_timestamp())
            .withColumn("source_file", F.input_file_name())
)

#use append mode because bronze layer is expected to insert only from source
stream_writer = df_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_zone + "/registered_users_bz") \
    .outputMode("append") \
    .queryName("registered_users_bz_ingestion_stream") 

stream_writer.trigger(availableNow = True).toTable(f"{catalog}.{db_name}.registered_users_bz")

# if once == True:
#    stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{db_name}.registered_users_bz")
# else:
#     stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{db_name}.registered_users_bz")
      

In [0]:
schema = "mac_address string, gym bigint, login double, logout double"

df_stream = (spark.readStream
            .format("cloudFiles")
            .schema(schema)
            .option("maxFilesPerTrigger", 1)
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .load(landing_zone + "/gym_logins_bz")
            .withColumn("load_time", F.current_timestamp())
            .withColumn("source_file", F.input_file_name())
)

#use append mode because bronze layer is expected to insert only from source
stream_writer = df_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_zone + "/gym_logins_bz") \
    .outputMode("append") \
    .queryName("gym_logins_bz_ingestion_stream")

stream_writer.trigger(availableNow = True).toTable(f"{catalog}.{db_name}.gym_logins_bz")

In [0]:
once = True

schema = schema = "key string, value string, topic string, partition bigint, offset bigint, timestamp bigint"
df_date_lookup = spark.table(f"{catalog}.{db_name}.date_lookup").select("date", "week_part")

df_stream = (spark.readStream
            .format("cloudFiles")
            .schema(schema)
            .option("maxFilesPerTrigger", 3)
            .option("cloudFiles.format", "json")
            .load(landing_zone + "/kafka_multiplex_bz")
            .withColumn("load_time", F.current_timestamp())
            .withColumn("source_file", F.input_file_name())
            .join(F.broadcast(df_date_lookup),
                  [F.to_date((F.col("timestamp")/1000).cast("timestamp")) == F.
                             col("date")],
                            "left")
            )
            
stream_writer = df_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_zone + "/kafka_multiplex_bz") \
    .outputMode("append") \
    .queryName("kafka_multiplex_bz_ingestion_stream") 



stream_writer.trigger(availableNow = True).toTable(f"{catalog}.{db_name}.kafka_multiplex_bz")

**Silver Layer**

In [0]:
print(f"Creating users table...", end='')
spark.sql(f"""CREATE OR REPLACE TABLE {catalog}.{db_name}.users_user(
        user_id bigint, 
        device_id bigint, 
        mac_address string,
        registration_timestamp timestamp
        )
        """)  
print("Done")

In [0]:
print(f"Creating gym_logs table...", end='')
spark.sql(f"""CREATE OR REPLACE TABLE {catalog}.{db_name}.gym_logs(
        mac_address string,
        gym bigint,
        login timestamp,                      
        logout timestamp
        )
        """) 
print("Done")

In [0]:
print(f"Creating user_profile table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.user_profile(
        user_id bigint, 
        dob DATE, 
        sex STRING, 
        gender STRING, 
        first_name STRING, 
        last_name STRING, 
        street_address STRING, 
        city STRING, 
        state STRING, 
        zip INT, 
        updated TIMESTAMP)
        """)  
print("Done")

In [0]:
print(f"Creating heart_rate table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.heart_rate(
        device_id LONG, 
        time TIMESTAMP, 
        heartrate DOUBLE, 
        valid BOOLEAN)
        """)
print("Done")

In [0]:
print(f"Creating user_bins table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.user_bins(
        user_id BIGINT, 
        age STRING, 
        sex STRING, 
        city STRING, 
        state STRING)
        """)  
print("Done")

In [0]:
print(f"Creating workouts table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.workouts(
        user_id INT, 
        workout_id INT, 
        time TIMESTAMP, 
        action STRING, 
        session_id INT)
        """)  
print("Done")

In [0]:
print(f"Creating completed_workouts table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.completed_workouts(
        user_id INT, 
        workout_id INT, 
        session_id INT, 
        start_time TIMESTAMP, 
        end_time TIMESTAMP)
        """)  
print("Done")

In [0]:
print(f"Creating workout_bpm table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.workout_bpm(
        user_id INT, 
        workout_id INT, 
        session_id INT,
        start_time TIMESTAMP, 
        end_time TIMESTAMP,
        time TIMESTAMP, 
        heartrate DOUBLE)
        """)  
print("Done")

In [0]:
print(f"Creating date_lookup table...", end='')
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.date_lookup(
        date date, 
        week int, 
        year int, 
        month int, 
        dayofweek int, 
        dayofmonth int, 
        dayofyear int, 
        week_part string)
        """)  
print("Done")

**USERS TABLE SILVER LAYER**

  - Idempotent - User cannot register again so ignore the duplicates and insert the new records
  - Spark Structured Streaming accepts append only sources. 
    - This is not a problem for silver layer streams because bronze layer is insert only
    - However, you may want to allow bronze layer deletes due to regulatory compliance 
  - Spark Structured Streaming throws an exception if any modifications occur on the table being used as a source
    - This is a problem for silver layer streaming jobs.
    - ignoreDeletes allows to delete records on partition column in the bronze layer without exception on silver layer streams 
  - Starting version is to allow you to restart your stream from a given version just in case you need it
    - startingVersion is only applied for an empty checkpoint
  - Limiting your input stream size is critical for running on limited capacity



In [0]:
once=True

startingversion=0



merge_query = f"""
MERGE INTO {catalog}.{db_name}.users_user a
USING users_delta b
ON a.user_id = b.user_id
WHEN NOT MATCHED THEN INSERT *

"""


def upserter(df_micro_batch, batch_id):
    df_micro_batch.createOrReplaceTempView("users_delta")
    df_micro_batch._jdf.sparkSession().sql(merge_query)




df_delta =(spark.readStream
    .option("startingVersion", startingversion)
    .option("ignoreDeletes", "true")
    .table(f"{catalog}.{db_name}.registered_users_bz")
    .selectExpr("user_id", "device_id", "mac_address", "cast(registration_timestamp as timestamp)")
    .withWatermark("registration_timestamp", "30 seconds")
    .dropDuplicates(["user_id", "device_id"])
)


stream_writer =(df_delta.writeStream
    .foreachBatch(upserter)
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_zone}/users_user")
    .queryName("users_upsert_stream")

)


stream_writer.trigger(availableNow=True).start()

In [0]:
display(df_delta)

In [0]:

once=True

startingVersion=0



merge_query = f"""
MERGE INTO {catalog}.{db_name}.gym_logs a
USING gym_logs_delta b
ON a.mac_address=b.mac_address AND a.gym=b.gym AND a.login=b.login
WHEN MATCHED AND b.logout > a.login AND b.logout > a.logout
    THEN UPDATE SET logout = b.logout
WHEN NOT MATCHED THEN INSERT *
"""



def upserter(df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView("gym_logs_delta")
        df_micro_batch._jdf.sparkSession().sql(merge_query)




df_delta = (spark.readStream
                    .option("startingVersion", startingVersion)
                    .option("ignoreDeletes", True)
                    .table(f"{catalog}.{db_name}.gym_logins_bz")
                    .selectExpr("mac_address", "gym", "cast(login as timestamp)", "cast(logout as timestamp)")
                    .withWatermark("login", "30 seconds")
                    .dropDuplicates(["mac_address", "gym", "login"])
            )


stream_writer = (df_delta.writeStream
                            .foreachBatch(upserter)
                            .outputMode("update")
                            .option("checkpointLocation", f"{checkpoint_zone}/gym_logs")
                            .queryName("gym_logs_upsert_stream")
                )



stream_writer.trigger(availableNow=True).start()




**USER_PROFILE SILVER LAYER**

  - Idempotent - Insert new record
    - Ignore deletes
    - Update user details when
      - 1. update_type in ("new", "append")
      - 2. current update is newer than the earlier

In [0]:
merge_query = f"""
MERGE INTO {catalog}.{db_name}.user_profile a
USING user_profile_cdc b
ON a.user_id=b.user_id
WHEN MATCHED AND a.updated < b.updated
    THEN UPDATE SET *
WHEN NOT MATCHED
    THEN INSERT *
"""


from pyspark.sql import functions as F
from pyspark.sql.window import Window

def upsert_user_profile_microbatch(df_micro_batch, batch_id):
    # Define a window to rank records per user_id by updated timestamp descending
    window = Window.partitionBy("user_id").orderBy(F.col("updated").desc())
    
    # Filter for only relevant update_types ("new", "update"), get latest per user_id
    df_filtered = (
        df_micro_batch.filter(F.col("update_type").isin(["new", "update"]))
        .withColumn("rank", F.rank().over(window))
        .filter("rank == 1")
        .drop("rank")
    )
    
    # Create or replace temp view for the MERGE SQL
    df_filtered.createOrReplaceTempView("user_profile_cdc")
    
    # Run the merge query to upsert into the Silver Delta table
    df_micro_batch._jdf.sparkSession().sql(merge_query)
    
    print(f"Batch {batch_id} processed.")





schema = """
    user_id bigint, update_type STRING, timestamp FLOAT, 
    dob STRING, sex STRING, gender STRING, first_name STRING, last_name STRING, 
    address STRUCT<street_address: STRING, city: STRING, state: STRING, zip: INT>
"""

df_cdc = (
    spark.readStream
         .option("startingVersion", 0)
         .option("ignoreDeletes", True)
         .table(f"{catalog}.{db_name}.kafka_multiplex_bz")
         .filter("topic = 'user_info'")
         .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
         .select("v.*")
         .select(
             "user_id",
             F.to_date("dob", "MM/dd/yyyy").alias("dob"),
             "sex", "gender", "first_name", "last_name",
             "address.*",
             F.col("timestamp").cast("timestamp").alias("updated"),
             "update_type"
         )
         .withWatermark("updated", "30 seconds")
         .dropDuplicates(["user_id", "updated"])
)


stream_writer = (
                df_cdc.writeStream
                    .foreachBatch(upsert_user_profile_microbatch)
                    .outputMode("update")
                    .option("checkpointLocation", f"{checkpoint_zone}/user_profile")
                    .queryName("user_profile_stream")

            )


stream_writer.trigger(availableNow=True).start()

**WORKOUTS SILVER LAYER**

 - Idempotent - User cannot have two workout sessions at the same time. So ignore the duplicates and insert the new records


In [0]:
startingVersion=0
merge_query = f"""
MERGE INTO {catalog}.{db_name}.workouts a
USING workouts_delta b
ON a.user_id=b.user_id AND a.time=b.time
WHEN NOT MATCHED THEN INSERT *
"""

def upserter(df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView("workouts_delta")
        df_micro_batch._jdf.sparkSession().sql(merge_query)

schema = "user_id INT, workout_id INT, timestamp FLOAT, action STRING, session_id INT"

df_delta = (spark.readStream
        .option("startingVersion", startingVersion)
        .option("ignoreDeletes", True)
        .table(f"{catalog}.{db_name}.kafka_multiplex_bz")
        .filter("topic = 'workout'")
        .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
        .select("v.*")
        .select("user_id", "workout_id", 
                F.col("timestamp").cast("timestamp").alias("time"), 
                "action", "session_id")
        .withWatermark("time", "30 seconds")
        .dropDuplicates(["user_id", "time"])
)



stream_writer = (df_delta.writeStream
                            .foreachBatch(upserter)
                            .outputMode("update")
                            .option("checkpointLocation",f"{checkpoint_zone}/workouts")
                            .queryName("workouts_upsert_stream")
                )




stream_writer.trigger(availableNow=True).start()

**HEART_RATES SILVER LAYER**

 - Idempotent - Only one BPM signal is allowed at a timestamp. So ignore the duplicates and insert the new records

In [0]:
merge_query = f"""
MERGE INTO {catalog}.{db_name}.heart_rate a
USING heart_rate_delta b
ON a.device_id=b.device_id AND a.time=b.time
WHEN NOT MATCHED THEN INSERT *
"""
def upserter(df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView("heart_rate_delta")
        df_micro_batch._jdf.sparkSession().sql(merge_query)

schema = "device_id LONG, time TIMESTAMP, heartrate DOUBLE"


df_delta = (spark.readStream
                    .option("startingVersion", startingVersion)
                    .option("ignoreDeletes", True)
                    .table(f"{catalog}.{db_name}.kafka_multiplex_bz")
                    .filter("topic = 'bpm'")
                    .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
                    .select("v.*", F.when(F.col("v.heartrate") <= 0, False).otherwise(True).alias("valid"))
                    .withWatermark("time", "30 seconds")
                    .dropDuplicates(["device_id", "time"])
            )


stream_writer = (df_delta.writeStream
                            .foreachBatch(upserter)
                            .outputMode("update")
                            .option("checkpointLocation", f"{checkpoint_zone}/heart_rate")
                            .queryName("heart_rate_upsert_stream")
                )

stream_writer.trigger(availableNow=True).start()



**AGE BIN FUNCTION**

In [0]:
from pyspark.sql.functions import floor, months_between, current_date, when, col

def age_bins(dob_col):
    age_col = floor(months_between(current_date(), dob_col) / 12)
    
    return (when(age_col < 18, "under 18")
              .when((age_col >= 18) & (age_col < 25), "18-25")
              .when((age_col >= 25) & (age_col < 35), "25-35")
              .when((age_col >= 35) & (age_col < 45), "35-45")
              .when((age_col >= 45) & (age_col < 55), "45-55")
              .when((age_col >= 55) & (age_col < 65), "55-65")
              .when((age_col >= 65) & (age_col < 75), "65-75")
              .when((age_col >= 75) & (age_col < 85), "75-85")
              .when((age_col >= 85) & (age_col < 95), "85-95")
              .when(age_col >= 95, "95+")
              .otherwise("invalid age"))

**USER BINS**

  - Idempotent - This table is maintained as SCD Type 1 dimension
  - Insert new user_id records 
  - Update old records using the user_id

In [0]:
merge_query = f"""
MERGE INTO {catalog}.{db_name}.user_bins a
USING user_bins_delta b
ON a.user_id=b.user_id
WHEN MATCHED 
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
"""

def upserter(df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView("user_bins_delta")
        df_micro_batch._jdf.sparkSession().sql(merge_query)

# Step 1: Load list of registered users from silver `users` table
df_user = spark.table(f"{catalog}.{db_name}.users_user").select("user_id")


# Step 2: Read streaming changes from `user_profile` table
# - Set `ignoreChanges=True` to track only new or updated records (no deletes)

df_delta = (
    spark.readStream
        .option("startingVersion", startingVersion)
        .option("ignoreChanges", True)
        .table(f"{catalog}.{db_name}.user_profile")
        .join(df_user, on="user_id", how="left")  # Statelss left join with static DataFrame
        .select(
            "user_id",
            age_bins(col("dob")).alias("age"),  # Use previously defined function
            "sex", 
            "city", 
            "state"
        )
)


stream_writer = (df_delta.writeStream
                          .foreachBatch(upserter)
                          .outputMode("update")
                          .option("checkpointLocation", f"{checkpoint_zone}/user_bin")
                          .queryName("user_bins_upsert_stream")
                )


stream_writer.trigger(availableNow=True).start()





**COMPLETED WORKOUTS**

In [0]:
#Idempotent - Only one user workout session completes. So ignore the duplicates and insert the new records
merge_query = f"""
MERGE INTO {catalog}.{db_name}.completed_workouts a
USING completed_workouts_delta b
ON a.user_id=b.user_id AND a.workout_id = b.workout_id AND a.session_id=b.session_id
WHEN NOT MATCHED THEN INSERT *
"""

def upserter(df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView("completed_workouts_delta")
        df_micro_batch._jdf.sparkSession().sql(merge_query)

df_start = (spark.readStream
                    .option("startingVersion", startingVersion)
                    .option("ignoreDeletes", True)
                    .table(f"{catalog}.{db_name}.workouts")
                    .filter("action = 'start'")                         
                    .selectExpr("user_id", "workout_id", "session_id", "time as start_time")
                    .withWatermark("start_time", "30 seconds")
            )

df_stop = (spark.readStream
                    .option("startingVersion", startingVersion)
                    .option("ignoreDeletes", True)
                    .table(f"{catalog}.{db_name}.workouts")
                    .filter("action = 'stop'")                         
                    .selectExpr("user_id", "workout_id", "session_id", "time as end_time")
                    .withWatermark("end_time", "30 seconds")
            )

# State cleanup - Define a condition to clean the state
#               - stop must occur within 3 hours of start 
#               - stop < start + 3 hours
join_condition = [df_start.user_id == df_stop.user_id, df_start.workout_id==df_stop.workout_id, df_start.session_id==df_stop.session_id, 
                    df_stop.end_time < df_start.start_time + F.expr('interval 3 hour')]         

df_delta = (df_start.join(df_stop, join_condition)
                    .select(df_start.user_id, df_start.workout_id, df_start.session_id, df_start.start_time, df_stop.end_time)
            )

stream_writer = (df_delta.writeStream
                            .foreachBatch(upserter)
                            .outputMode("append")
                            .option("checkpointLocation", f"{checkpoint_zone}/completed_workouts")
                            .queryName("completed_workouts_upsert_stream")
                )


stream_writer.trigger(availableNow=True).start()

**WORKOUT BPM**

 - Idempotent - Only one user workout session completes. So ignore the duplicates and insert the new records

In [0]:
from pyspark.sql.functions import expr
#Idempotent - Only one user workout session completes. So ignore the duplicates and insert the new records
merge_query = f"""
MERGE INTO {catalog}.{db_name}.workout_bpm a
USING workout_bpm_delta b
ON a.user_id=b.user_id AND a.workout_id = b.workout_id AND a.session_id=b.session_id AND a.time=b.time
WHEN NOT MATCHED THEN INSERT *
"""

def upserter(df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView("workout_bpm_delta")
        df_micro_batch._jdf.sparkSession().sql(merge_query)

# Load static user table
df_users = spark.read.table(f"{catalog}.{db_name}.users_user")

# Load the completed workouts stream
df_completed_workouts = (
    spark.readStream
         .option("startingVersion", 0)
         .option("ignoreDeletes", True)
         .table(f"{catalog}.{db_name}.completed_workouts")
         .join(df_users, "user_id")
         .selectExpr("user_id", "device_id", "workout_id", "session_id", "start_time", "end_time")
         .withWatermark("end_time", "30 seconds")
)

# Load the heart rate stream
df_bpm = (
    spark.readStream
         .option("startingVersion", 0)
         .option("ignoreDeletes", True)
         .table(f"{catalog}.{db_name}.heart_rate")
         .filter("valid = True")
         .selectExpr("device_id", "time", "heartrate")
         .withWatermark("time", "30 seconds")
)

# Define the join condition between BPM and workouts
join_condition = [
    df_completed_workouts.device_id == df_bpm.device_id,
    df_bpm.time > df_completed_workouts.start_time,
    df_bpm.time <= df_completed_workouts.end_time,
    df_completed_workouts.end_time < df_bpm.time + expr("interval 3 hours")
]

# Join and select the desired columns
df_delta = (
    df_bpm.join(df_completed_workouts, join_condition)
          .select("user_id", "workout_id", "session_id", "start_time", "end_time", "time", "heartrate")
)

stream_writer = (
    df_delta.writeStream
            .foreachBatch(upserter)
            .outputMode("append")
            .option("checkpointLocation", f"{checkpoint_zone}/workout_bpms")
            .queryName("workout_bpm_upsert_stream")

)



stream_writer.trigger(availableNow=True).start()

**GOLD LAYER**

In [0]:
print(f"Creating workout_bpm_summary table...", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.workout_bpm_summary (
        workout_id INT, 
        session_id INT, 
        user_id BIGINT, 
        age STRING, 
        sex STRING, 
        city STRING, 
        state STRING, 
        min_bpm DOUBLE, 
        avg_bpm DOUBLE, 
        max_bpm DOUBLE, 
        num_recordings BIGINT
    )
""")
print("Done")

**WORKOUT_BPM_SUMMARY**

  - Idempotent - Once a workout session is complete, It doesn't change. So insert only the new records

In [0]:
merge_query = f"""
MERGE INTO {catalog}.{db_name}.workout_bpm_summary a
USING workout_bpm_summary_delta b
ON a.user_id=b.user_id AND a.workout_id = b.workout_id AND a.session_id=b.session_id
WHEN NOT MATCHED THEN INSERT *
"""


def upserter(df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView("workout_bpm_summary_delta")
        df_micro_batch._jdf.sparkSession().sql(merge_query)




df_users = spark.read.table(f"{catalog}.{db_name}.user_bins")

df_delta = (spark.readStream
                  .option("startingVersion", startingVersion)
                  .table(f"{catalog}.{db_name}.workout_bpm")
                  .withWatermark("end_time", "30 seconds")
                  .groupBy("user_id", "workout_id", "session_id", "end_time")
                  .agg(F.min("heartrate").alias("min_bpm"), F.mean("heartrate").alias("avg_bpm"), 
                      F.max("heartrate").alias("max_bpm"), F.count("heartrate").alias("num_recordings"))                         
                  .join(df_users, ["user_id"])
                  .select("workout_id", "session_id", "user_id", "age", "sex", "city", "state", "min_bpm", "avg_bpm", "max_bpm", "num_recordings")
              )


stream_writer = (df_delta.writeStream
                  .foreachBatch(upserter)
                  .outputMode("append")
                  .option("checkpointLocation", f"{checkpoint_zone}/workout_bpm_summary")
                  .queryName("workout_bpm_summary_upsert_stream")
          )


stream_writer.trigger(availableNow=True).start()

In [0]:
display(df_delta)

In [0]:
workout = spark.read.table(f"{catalog}.{db_name}.workout_bpm_summary")
display(workout)